# Task 1: Data Processing & Preparation

## Project Overview

**Research Question:** Can we predict traffic congestion on Vinh Tuy Bridge (Hanoi) based on temporal and weather features?

**Dataset:** Self-collected via TomTom Routing API (traffic data) and Visual Crossing API (weather data) using stratified hour sampling across 10 representative time slots per day.

**Target variable:** `is_congested` — a binary label (1 = congested, 0 = normal flow), determined by whether the live speed ratio drops below 80% of free-flow speed.

**Pipeline in this notebook:**
1. Load raw dataset
2. Data Cleaning (handle missing values, derive time features)
3. Feature Engineering (create 6 golden features)
4. Data Visualization (understand distributions and relationships)
5. Canonical Train/Test Split (80/20, saved to disk for use in Task 2 & 3)

## Section 1: Imports

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.model_selection import train_test_split

# Resolve paths relative to this notebook's location
NOTEBOOK_DIR = os.path.abspath('')
DATASETS_DIR = os.path.join(NOTEBOOK_DIR, '..', '..', 'Datasets')

sns.set_theme(style='whitegrid')
print(f'Datasets directory: {os.path.abspath(DATASETS_DIR)}')

## Section 2: Load Raw Dataset

The raw CSV (`VINH_TUY.csv`) was produced by the API scraper. Each row represents one traffic measurement at a specific timestamp, direction (Inbound / Outbound), and hour, along with simultaneous weather conditions fetched from Visual Crossing.

Key raw columns include: `timestamp`, `direction`, `current_speed`, `free_flow_time_s`, `speed_ratio_proxy`, `is_congested`, `rain_mm`, `visibility`, `temp`, `humidity`.

In [ ]:
raw_path = os.path.join(DATASETS_DIR, 'VINH_TUY.csv')
df_raw = pd.read_csv(raw_path)

print(f'Raw dataset shape: {df_raw.shape}')
print(f'\nColumns: {list(df_raw.columns)}')
df_raw.head()

## Section 3: Data Cleaning

The cleaning pipeline handles four issues:

1. **Missing target variable:** Rows where `is_congested` is null are dropped immediately — they cannot be used in any learning task.

2. **Time feature derivation:** We extract `hour_of_day`, `day_of_week` (0=Monday, 6=Sunday), and `is_holiday` from the raw `timestamp` column. These are necessary for feature engineering in the next step.

3. **Missing value imputation — Grouped Median Strategy:** Instead of a global median fill, we impute missing numeric values using medians grouped by `(direction, day_of_week, hour_of_day)`. This preserves temporal and spatial patterns — e.g., inbound traffic on Monday mornings has a very different speed profile than outbound traffic on Sunday afternoons. A global median would flatten these differences and introduce bias.

4. **Deduplication:** Duplicate rows are removed to prevent the model from overfitting to repeated identical observations.

In [ ]:
import holidays

TARGET_COL = 'is_congested'
df = df_raw.copy()
initial_rows = len(df)

# 1. Drop rows without target
if TARGET_COL in df.columns:
    df = df.dropna(subset=[TARGET_COL])
    df[TARGET_COL] = df[TARGET_COL].astype(int)

# 2. Derive time features
vn_holidays = holidays.VN()
if 'timestamp' in df.columns:
    df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
    df['hour_of_day'] = df['timestamp'].dt.hour
    df['day_of_week'] = df['timestamp'].dt.dayofweek
    df['is_holiday'] = df['timestamp'].dt.date.apply(lambda d: d in vn_holidays).astype(int)

# 3. Grouped-median imputation
num_cols = df.select_dtypes(include=[np.number]).columns
if all(c in df.columns for c in ['direction', 'day_of_week', 'hour_of_day']):
    group_cols = ['direction', 'day_of_week', 'hour_of_day']
    df[num_cols] = df.groupby(group_cols)[num_cols].transform(lambda x: x.fillna(x.median()))
if 'direction' in df.columns and 'hour_of_day' in df.columns:
    df[num_cols] = df.groupby(['direction', 'hour_of_day'])[num_cols].transform(lambda x: x.fillna(x.median()))
for col in num_cols:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())

# 4. Categorical fallback
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].fillna('Unknown')

# 5. Remove duplicates
df = df.drop_duplicates()

print(f'Rows before cleaning: {initial_rows}')
print(f'Rows after cleaning:  {len(df)}')

# Save cleaned dataset
cleaned_path = os.path.join(DATASETS_DIR, 'Cleaned Dataset.csv')
df.to_csv(cleaned_path, index=False)
print(f'\nCleaned dataset saved to: {cleaned_path}')
df.head()

## Section 4: Feature Engineering

We engineer exactly **6 features** to feed into both the supervised and unsupervised models. These were selected to be maximally informative while eliminating multicollinearity.

| Feature | Type | Rationale |
|---|---|---|
| `is_rush_hour` | Binary | Hours 7, 8, 16, 17, 18 are peak commute times in Hanoi — the strongest known driver of congestion |
| `direction_inbound` | Binary | Inbound (toward city center) has a fundamentally different congestion pattern than outbound |
| `adverse_weather_score` | Ordinal (0/1/2) | A 3-tier heuristic: 0 = clear, 1 = light rain or low visibility (< 2km), 2 = heavy rain (≥7.6mm) or dense fog (≤400m) |
| `rush_weather_interaction` | Continuous | Multiplicative synergy: `is_rush_hour × adverse_weather_score`. Rush-hour + bad weather has a compounding effect on congestion beyond either factor alone |
| `dow_sin` | Continuous | Sine of day-of-week × (2π/7). **Why trigonometric encoding?** Raw day numbers (0–6) imply Sunday=6 is "far" from Monday=0, but they wrap around cyclically. Sine/cosine encoding preserves the circular nature of the weekly cycle, preventing models from being misled by the arbitrary numeric gap. |
| `dow_cos` | Continuous | Cosine component — together with `dow_sin`, uniquely encodes every day of the week on a circle |

**Final selection:** Only these 6 features + the target are kept in the processed dataset, discarding raw speed, travel time, and temperature columns that would introduce data leakage or noise.

In [ ]:
# 1. Rush hour flag
if 'hour_of_day' in df.columns:
    df['is_rush_hour'] = df['hour_of_day'].isin([7, 8, 16, 17, 18]).astype(int)

# 2. Direction encoding
if 'direction' in df.columns:
    df['direction_inbound'] = df['direction'].apply(lambda x: 1 if 'inbound' in str(x).lower() else 0)

# 3. Adverse weather score
def calculate_adverse_weather(row):
    rain = row.get('rain_mm', 0)
    vis  = row.get('visibility', 10000)
    if rain >= 7.6 or vis <= 400:  return 2  # Severe
    elif rain > 0  or vis < 2000:  return 1  # Mild adverse
    return 0                                  # Clear

if 'rain_mm' in df.columns and 'visibility' in df.columns:
    df['adverse_weather_score'] = df.apply(calculate_adverse_weather, axis=1)
    if 'is_rush_hour' in df.columns:
        df['rush_weather_interaction'] = df['is_rush_hour'] * df['adverse_weather_score']

# 4. Cyclical day-of-week encoding
if 'day_of_week' in df.columns:
    df['dow_sin'] = np.sin(df['day_of_week'] * (2 * np.pi / 7))
    df['dow_cos'] = np.cos(df['day_of_week'] * (2 * np.pi / 7))

# 5. Keep only 7 final columns
golden_features = ['is_congested', 'is_rush_hour', 'direction_inbound',
                   'adverse_weather_score', 'rush_weather_interaction', 'dow_sin', 'dow_cos']
df = df.dropna()
existing_golden = [c for c in golden_features if c in df.columns]
df_processed = df[existing_golden]

# Save processed dataset
processed_path = os.path.join(DATASETS_DIR, 'Processed Dataset.csv')
df_processed.to_csv(processed_path, index=False)
print(f'Processed dataset: {df_processed.shape[0]} rows × {df_processed.shape[1]} columns')
print(f'Columns: {list(df_processed.columns)}')
print(f'Saved to: {processed_path}')
df_processed.head()

## Section 5: Data Visualization

The following four charts explore feature distributions, relationships, and correlations to support our analytical choices in Tasks 2 and 3.

- **Plot 1 — Class Distribution:** Shows the imbalance between normal flow and congested records. This justifies why we report F1-Score alongside accuracy in Task 2 — accuracy alone would be misleading with imbalanced classes.
- **Plot 2 — Weather Severity vs. Congestion:** Validates our custom `adverse_weather_score` heuristic. If congestion probability increases monotonically with the score, the heuristic is effective.
- **Plot 3 — Rush Hour × Weather Interaction:** Demonstrates the synergistic compounding effect that motivated the `rush_weather_interaction` feature.
- **Plot 4 — Correlation Heatmap:** Verifies that our 6 golden features have low pairwise correlation (no severe multicollinearity), which is important for the stability of logistic regression coefficients.

In [ ]:
# Plot 1: Class Distribution
plt.figure(figsize=(8, 6))
custom_colors = ['#4A90E2', '#D9534F']
ax = sns.countplot(x='is_congested', data=df_processed, palette=custom_colors)
plt.title('Distribution of Traffic Congestion States', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Traffic Flow State', fontsize=12, labelpad=10)
plt.ylabel('Number of Records', fontsize=12, labelpad=10)
ax.set_xticks([0, 1])
ax.set_xticklabels(['Normal Flow', 'Congested'])
blue_patch = mpatches.Patch(color='#4A90E2', label='0: Normal Flow')
red_patch  = mpatches.Patch(color='#D9534F', label='1: Congested Traffic')
ax.legend(handles=[blue_patch, red_patch], title='Traffic Status', loc='upper right')
for p in ax.patches:
    h = p.get_height()
    if h > 0:
        ax.annotate(f'{int(h)}', (p.get_x() + p.get_width() / 2., h),
                    ha='center', va='bottom', fontsize=11, xytext=(0, 5), textcoords='offset points')
plt.tight_layout()
plt.show()
print('Plot 1: Target Distribution')

In [ ]:
# Plot 2: Congestion Probability by Weather Severity
if 'adverse_weather_score' in df_processed.columns:
    plt.figure(figsize=(8, 6))
    weather_colors = ['#8FBC8F', '#F4A460', '#D9534F']
    ax = sns.barplot(x='adverse_weather_score', y='is_congested', data=df_processed,
                     palette=weather_colors, errorbar=None)
    plt.title('Congestion Probability by Weather Severity', fontsize=14, fontweight='bold', pad=15)
    plt.xlabel('Weather Severity Score', fontsize=12, labelpad=10)
    plt.ylabel('Probability of Congestion', fontsize=12, labelpad=10)
    plt.ylim(0, 1)
    ax.set_xticks([0, 1, 2])
    ax.set_xticklabels(['Clear (0)', 'Mild (1)', 'Severe (2)'])
    g_patch = mpatches.Patch(color='#8FBC8F', label='0: Clear / Normal')
    o_patch = mpatches.Patch(color='#F4A460', label='1: Light Rain / Low Visibility')
    r_patch = mpatches.Patch(color='#D9534F', label='2: Heavy Rain / Dense Fog')
    ax.legend(handles=[g_patch, o_patch, r_patch], title='Weather Severity', loc='upper left')
    plt.tight_layout()
    plt.show()
    print('Plot 2: Weather Impact on Congestion')

In [ ]:
# Plot 3: Interaction Effect — Rush Hour × Weather
if 'is_rush_hour' in df_processed.columns and 'adverse_weather_score' in df_processed.columns:
    plt.figure(figsize=(10, 6))
    weather_colors = ['#8FBC8F', '#F4A460', '#D9534F']
    ax = sns.barplot(x='is_rush_hour', y='is_congested', hue='adverse_weather_score',
                     data=df_processed, palette=weather_colors, errorbar=None)
    plt.title('Interaction Effect: Rush Hour vs. Weather Severity', fontsize=14, fontweight='bold', pad=15)
    plt.xlabel('Rush Hour Period', fontsize=12, labelpad=10)
    plt.ylabel('Probability of Congestion', fontsize=12, labelpad=10)
    plt.ylim(0, 1)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Off-Peak Hour', 'Rush Hour'])
    g_patch = mpatches.Patch(color='#8FBC8F', label='0: Clear / Normal')
    o_patch = mpatches.Patch(color='#F4A460', label='1: Light Rain / Low Visibility')
    r_patch = mpatches.Patch(color='#D9534F', label='2: Heavy Rain / Dense Fog')
    ax.legend(handles=[g_patch, o_patch, r_patch], title='Weather Severity', loc='upper left')
    plt.tight_layout()
    plt.show()
    print('Plot 3: Interaction Effect')

In [ ]:
# Plot 4: Pearson Correlation Heatmap
features_to_plot = ['is_congested', 'is_rush_hour', 'direction_inbound',
                    'adverse_weather_score', 'rush_weather_interaction', 'dow_sin', 'dow_cos']
existing = [c for c in features_to_plot if c in df_processed.columns]
corr_matrix = df_processed[existing].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, square=True, linewidths=.5, cbar_kws={'shrink': .8})
plt.title('Pearson Correlation Matrix (Golden Features)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print('Plot 4: Correlation Heatmap')

## Section 6: Canonical Train/Test Split & Dataset Export

The project rubric requires that the **same** training and testing datasets be used consistently across supervised (Task 2) and unsupervised (Task 3) learning. Performing the split here — once — guarantees this.

**Why 80/20?**
The rubric mandates at least 50 valid samples per set. Our dataset is large enough that a 20% test set comfortably exceeds this threshold while still leaving 80% of the data to maximize the model's learning signal. A larger test set (e.g., 50%) would waste training data; a smaller one would give less reliable performance estimates.

**`stratify=y`:** Ensures the class ratio of `is_congested` is preserved in both the train and test sets. Without this, random splits could place disproportionately more congested samples in one set, biasing evaluation.

**`random_state=42`:** Fixes the random seed for full reproducibility. Anyone re-running this notebook will get the exact same split.

The resulting `Train Dataset.csv` and `Test Dataset.csv` are the **single source of truth** loaded by Task 2 and Task 3 notebooks.

In [ ]:
features = ['is_rush_hour', 'direction_inbound', 'adverse_weather_score',
            'rush_weather_interaction', 'dow_sin', 'dow_cos']
target = 'is_congested'

X = df_processed[features]
y = df_processed[target].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Reassemble full DataFrames (features + target) for saving
df_train = X_train.copy()
df_train[target] = y_train.values

df_test = X_test.copy()
df_test[target] = y_test.values

# Save to Datasets/
train_path = os.path.join(DATASETS_DIR, 'Train Dataset.csv')
test_path  = os.path.join(DATASETS_DIR, 'Test Dataset.csv')
df_train.to_csv(train_path, index=False)
df_test.to_csv(test_path, index=False)

print('=== CANONICAL SPLIT SUMMARY ===')
print(f'Full dataset:   {len(df_processed):>6} rows')
print(f'Training set:   {len(df_train):>6} rows  ({len(df_train)/len(df_processed)*100:.1f}%)')
print(f'Testing set:    {len(df_test):>6} rows  ({len(df_test)/len(df_processed)*100:.1f}%)')
print()
print(f'Train congestion rate: {y_train.mean()*100:.2f}%')
print(f'Test  congestion rate: {y_test.mean()*100:.2f}%  ← stratification verified')
print()
print(f'Train Dataset saved: {train_path}')
print(f'Test Dataset saved:  {test_path}')